In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys, os, re, json, math, random
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/TLM_V03')
assert DRIVE_ROOT.exists(), f"{DRIVE_ROOT} not found -- upload the project to Drive first"
sys.path.insert(0, str(DRIVE_ROOT))

%pip install -q --force-reinstall "transformers==4.44.2"

from src.codet5 import (
    TASK_TOKEN_NAME, SRTextDataset, collate_sr,
    hf_run_epoch, hf_validate, hf_greedy_decode, decode_target_text,
    parse_sr_pairs_text, apply_sr_text, MAX_NEW_TOKENS,
)
from src.training import make_scheduler

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, T5ForConditionalGeneration
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")

In [ ]:
BRIDGING_DIR   = DRIVE_ROOT / "data" / "bridging"
TRAIN_PATH = BRIDGING_DIR / "bridging_train.jsonl"
VAL_PATH   = BRIDGING_DIR / "bridging_val.jsonl"

RUN_NAME = "codet5_official_bridging"
BRIDGING_CKPT_DIR = DRIVE_ROOT / "data" / "checkpoints" / RUN_NAME
BRIDGING_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BRIDGING_CKPT_DIR={BRIDGING_CKPT_DIR}")
for p in (TRAIN_PATH, VAL_PATH):
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")

In [ ]:
TASK_MARKER_FMT = "<{}>"
SEARCH_MARKER   = "<search>"
REPLACE_MARKER  = "<replace>"

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")

model = T5ForConditionalGeneration.from_pretrained("Salesforce/codet5-small").to(device)

new_tokens = [TASK_MARKER_FMT.format(name) for name in TASK_TOKEN_NAME.values()] + [SEARCH_MARKER, REPLACE_MARKER]
num_added = tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
model.resize_token_embeddings(len(tokenizer))
print(f"Added {num_added} new tokens: {new_tokens}")
print(f"Vocab size (incl. new tokens): {len(tokenizer)}")

PAD_ID = tokenizer.pad_token_id
print(f"pad_token={tokenizer.pad_token!r} id={PAD_ID}  |  "
      f"eos_token={tokenizer.eos_token!r} id={tokenizer.eos_token_id}  |  "
      f"bos_token={tokenizer.bos_token!r} id={tokenizer.bos_token_id}  |  "
      f"decoder_start_token_id={model.config.decoder_start_token_id}")

n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.3f}M")

model.gradient_checkpointing_enable()
model.config.use_cache = False

In [ ]:
train_ds = SRTextDataset(TRAIN_PATH, tokenizer, TASK_MARKER_FMT, SEARCH_MARKER, REPLACE_MARKER)
val_ds   = SRTextDataset(VAL_PATH, tokenizer, TASK_MARKER_FMT, SEARCH_MARKER, REPLACE_MARKER)
print(f"train: {len(train_ds):,} kept, {train_ds.dropped} dropped {train_ds.dropped_by_type}")
print(f"val:   {len(val_ds):,} kept, {val_ds.dropped} dropped {val_ds.dropped_by_type}")

BATCH_SIZE       = 16

GRAD_ACCUM_STEPS = 2
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
_PIN        = torch.cuda.is_available()
_collate    = lambda b: collate_sr(b, PAD_ID)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=_collate, num_workers=NUM_WORKERS, pin_memory=_PIN)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=_collate, num_workers=NUM_WORKERS, pin_memory=_PIN)
print(f"train batches: {len(train_loader)}  |  val batches: {len(val_loader)}  |  "
      f"effective batch size: {EFFECTIVE_BATCH_SIZE}  |  device: {device}")

In [ ]:
EPOCHS       = 4
LR           = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
GRAD_CLIP    = 1.0
PATIENCE     = 2
USE_AMP      = False
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_available()
                               and torch.cuda.is_bf16_supported()) else None

steps_per_epoch = len(train_loader) // GRAD_ACCUM_STEPS
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = make_scheduler(optimizer, warmup_steps, total_steps)

print(f"epochs={EPOCHS}  steps/epoch={steps_per_epoch}  total={total_steps}  "
      f"warmup={warmup_steps}  lr={LR}  effective_batch={EFFECTIVE_BATCH_SIZE}")

In [ ]:
RESUME_PATH = BRIDGING_CKPT_DIR / "last.pt"

best_val = float("inf")
best_ckpt_path = None
bad_epochs = 0
history = []
start_epoch = 1

if RESUME_PATH.exists():
    print(f"Resuming from {RESUME_PATH}")
    _ckpt = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(_ckpt["model"])
    optimizer.load_state_dict(_ckpt["optimizer"])
    scheduler.load_state_dict(_ckpt["scheduler"])
    start_epoch = _ckpt["epoch"] + 1
    best_val = _ckpt["best_val"]
    bad_epochs = _ckpt["bad_epochs"]
    best_ckpt_path = Path(_ckpt["best_ckpt_path"]) if _ckpt.get("best_ckpt_path") else None
    history = _ckpt["history"]
    print(f"  resuming at epoch {start_epoch}  |  best_val so far {best_val:.4f}  |  bad_epochs {bad_epochs}/{PATIENCE}")
else:
    print("No last.pt found -- starting fresh bridging run")

for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{EPOCHS} ===")
    tr_loss = hf_run_epoch(model, train_loader, optimizer, scheduler, PAD_ID, device,
                           grad_accum_steps=GRAD_ACCUM_STEPS, amp_dtype=AMP_DTYPE, grad_clip=GRAD_CLIP)
    va_loss = hf_validate(model, val_loader, PAD_ID, device, amp_dtype=AMP_DTYPE)
    print(f"epoch {epoch}: train_loss {tr_loss:.4f}  val_loss {va_loss:.4f}")
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss})

    if va_loss < best_val:
        best_val = va_loss
        bad_epochs = 0
        best_ckpt_path = BRIDGING_CKPT_DIR / f"{RUN_NAME}_epoch_{epoch}_val{va_loss:.4f}.pt"
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": va_loss,
                    "vocab_size": len(tokenizer)}, best_ckpt_path)
        print(f"  saved best -> {best_ckpt_path.name}")
    else:
        bad_epochs += 1
        print(f"  no improvement ({bad_epochs}/{PATIENCE})")

    torch.save({
        "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "epoch": epoch,
        "best_val": best_val, "bad_epochs": bad_epochs,
        "best_ckpt_path": str(best_ckpt_path) if best_ckpt_path else None,
        "history": history,
    }, RESUME_PATH)

    if bad_epochs >= PATIENCE:
        print("early stopping.")
        break

print(f"\nbest val_loss: {best_val:.4f}")


In [ ]:
import matplotlib.pyplot as plt

history_path = BRIDGING_CKPT_DIR / "training_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"history saved -> {history_path}")

epochs_x   = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss   = [h["val_loss"] for h in history]
best_ep    = epochs_x[val_loss.index(min(val_loss))]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(epochs_x, train_loss, marker="o", label="train loss")
ax.plot(epochs_x, val_loss, marker="o", label="val loss")
ax.axvline(x=best_ep, color="green", linestyle="--", alpha=0.5, label=f"best (ep {best_ep})")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.set_title(f"{RUN_NAME} ({n_params/1e6:.1f}M params) -- bridging loss curves")
ax.legend()
plt.tight_layout()
plt.savefig(BRIDGING_CKPT_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
assert best_ckpt_path is not None, "no checkpoint saved -- did training run?"
_best_state = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(_best_state["model"])
model.eval()
model.config.use_cache = True
print(f"loaded best: {best_ckpt_path.name}  val_loss={_best_state['val_loss']:.4f}")

In [ ]:
from collections import defaultdict

@torch.no_grad()
def evaluate_bridging_applied_ok(model, tokenizer, val_path, task_marker_fmt, search_marker,
                                  replace_marker, n_per_type=100, seed=0):
    by_type = defaultdict(list)
    with open(val_path, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            by_type[d["type"]].append(d)
    rng = random.Random(seed)
    results = {}
    for ttype, recs in sorted(by_type.items()):
        sample = rng.sample(recs, min(n_per_type, len(recs)))
        ok = trunc = total = 0
        for d in sample:
            enc_text = task_marker_fmt.format(ttype) + d["before"]
            enc_ids = tokenizer(enc_text, truncation=True, max_length=512).input_ids
            pred_ids = hf_greedy_decode(model, tokenizer, enc_ids, device)
            pred_text, is_trunc = decode_target_text(tokenizer, pred_ids)
            hunks = parse_sr_pairs_text(pred_text, search_marker, replace_marker)
            pred_after = apply_sr_text(d["before"], hunks)
            total += 1
            trunc += int(is_trunc)
            if pred_after is not None and pred_after == d["after"]:
                ok += 1
        results[ttype] = (ok, total, trunc)
    return results

res = evaluate_bridging_applied_ok(model, tokenizer, VAL_PATH, TASK_MARKER_FMT,
                                   SEARCH_MARKER, REPLACE_MARKER, n_per_type=100)
print(f"{'task':18}{'applied_ok':>12}{'truncated':>11}")
oks = tots = 0
for t, (ok, tot, trunc) in res.items():
    oks += ok
    tots += tot
    print(f"{t:18}{ok:4d}/{tot:<4d} {ok/max(tot, 1)*100:5.1f}%   {trunc:3d}/{tot:<3d}")
print(f"{'OVERALL':18}{oks:4d}/{tots:<4d} {oks/max(tots, 1)*100:5.1f}%")
